# F1-scientific-python — Practice p13 — Solution

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
grid = (rng.random((12, 16)) < 0.35).astype(np.int64)   # ~35% of seats taken
print(grid.shape, "taken seats:", grid.sum())

**Task A — pad.** Build **`padded`**, a `(14, 18)` array of zeros with `grid`
sitting in its center (one extra border row/column of zeros on every side).
Use slice **assignment** — create the zeros, then assign `grid` into the
middle block.

Why this helps: with a zero border, every original seat has a full 3×3
neighborhood inside `padded`, so no edge cases remain.

In [ ]:
padded = np.zeros((14, 18), dtype=np.int64)
padded[1:13, 1:17] = grid
print(padded.shape, padded.sum() == grid.sum())

The border cells stay 0, so they contribute nothing when summed — edge seats simply see zero-filled neighbors.

**Task B — count neighbors, loop-free.** Implement exactly
`def neighbor_counts(padded):` returning a `(12, 16)` integer array whose
entry `(i, j)` is the number of taken seats among the 8 neighbors of seat
`(i, j)` in the original grid.

Hint: each of the 8 neighbor directions is one **shifted `(12, 16)` slice** of
`padded` (for example, the "up" neighbors of every seat at once are
`padded[0:12, 1:17]`). Add the eight slices elementwise. No loops.

In [ ]:
def neighbor_counts(padded):
    return (padded[0:12, 0:16] + padded[0:12, 1:17] + padded[0:12, 2:18]
            + padded[1:13, 0:16] +                    padded[1:13, 2:18]
            + padded[2:14, 0:16] + padded[2:14, 1:17] + padded[2:14, 2:18])

neighbors = neighbor_counts(padded)
print(neighbors.shape)
print(neighbors[:3, :8])

Each slice is the whole grid's view of one neighbor direction: shifting the `(12, 16)` window around the padded array by one step in each of the 8 directions. Adding them elementwise counts, for every seat simultaneously, how many of its 8 neighbors are 1 — a quadruple-nested loop's worth of work in one expression.

**Task C — mask questions, loop-free.** Using `neighbors` and `grid`:

- **`lonely`** — how many *taken* seats have zero taken neighbors
- **`crowded`** — how many *taken* seats have 4 or more taken neighbors
- **`quiet_empty`** — how many *empty* seats have no taken neighbors at all

In [ ]:
taken = grid == 1
lonely = (taken & (neighbors == 0)).sum()
crowded = (taken & (neighbors >= 4)).sum()
quiet_empty = ((~taken) & (neighbors == 0)).sum()
print("lonely:", lonely, "| crowded:", crowded, "| quiet empty:", quiet_empty)

Combine the occupancy mask with neighbor-count masks using `&` and `~`; each `.sum()` counts the seats satisfying both conditions at once.

### Answer check

In [ ]:
# Reference implementation for verification only (the loop ban applies to the
# solution above; an independent loop-based cross-check is the most convincing
# way to verify it).
ref = np.zeros_like(grid)
for i in range(12):
    for j in range(16):
        total = 0
        for di in (-1, 0, 1):
            for dj in (-1, 0, 1):
                if di == 0 and dj == 0:
                    continue
                ii, jj = i + di, j + dj
                if 0 <= ii < 12 and 0 <= jj < 16:
                    total += grid[ii, jj]
        ref[i, j] = total

assert padded.shape == (14, 18)
assert padded.sum() == grid.sum()
assert np.array_equal(padded[1:13, 1:17], grid)
assert neighbors.shape == (12, 16)
assert np.array_equal(neighbors, ref)
assert lonely == ((grid == 1) & (ref == 0)).sum()
assert crowded == ((grid == 1) & (ref >= 4)).sum()
assert quiet_empty == ((grid == 0) & (ref == 0)).sum()
print("all checks passed")